# Step 2 - Prototype 1

Here we mainly want to focus on adding the option to have trains going in both directions. In this context, we might have to implement dwell times (to handle buffering) and decide if tracks (all or only some) are directed or undirected. Additionally, we might have to rework the train schedules (more specifically their formulation) to some extent. All in all, this will set us up for longer time horizons.

Also, we might introduce train and station (passenger) weights as more realistic constraints here could potentially solve (or at least mitigate) lazy solver behavior. This could then allow us to get rid of the 'reach station by the end of the time horizon constraint', which might be somewhat problematic in the context of longer time horizons and train schedules with back and forth plans.

To get started, we use the code from the cleaned file for step 1 and iteratively modify it.

### The New Game Plan

We have more or less implemented the logic to allow for bidirectional traveling of trains in our network (still only tracks in a straight line). There are just some more wrinkles regarding longer horizons and multiple back and forth journies. These will be addressed and fixed soon though.

The needed changes and implementations to be made in this phase are:

#### Model Objective

We want to compute a weighted delay at all stations, for now just using the number of stations on a journey and the trains maximum capacity. In theory, this could be made more 'dynamic' to allow for station (and even time) specific amounts of passengers getting on and off. Possibly, this could also be turned into a passenger optimization problem with the network 'as the means'... This however will certainly not be part of our project xD.

#### Long Horizon

Possibly the most important part of allowing for longer time horizons is the ability to study delay propagation over multiple, partially returning / 'oscillating' journies.

In this regard, we have to fix our departure walls (keeping trains from leaving stations before the schedules departure time). Currently, visiting a station more than once (in total) whilst going in the same direction...

#### Variable Maximum Speeds

This is the other big topic. We would like to have trains of different speeds in our network. Currently, however we have the issue that our movement and uniqueness constraints are logically incompatible for speeds larger than one. To fix this, we will first try to fine-graine our time steps (sync with the fastest train in the network s.t. it travels one block per time unit). Other trains will then have wait times after entering a new block to implement this speed difference.

To make speed ratios more 'freely definable', we might use the smallest common multiple of our different train speeds. Here we'll have to assess the performance trade-off though.

In combination with this, we want to implement a general dwell time at stations as this is in a sense logically similar to the block dwell times. This could probably even be handled in the same loop while checking for `is_station`.

#### Safety distance

Trains should maintain a safety distance (especially important if we were to implement junctions later on), possibly dependent of their speed. This might also in a sense be similar to dwell times (as we have to wait), however, this also has some more dynamic components which we probably have to implement separately.

#### The Wrinkles

1. At the moment, trains are technically allowed to turn at any station. This will probably not be a problem though, as long as we are using a good model objective. Nonetheless, a fix might be implemented by only allowing direction switching at the `turn` stations that we are already computing for the current model objective.
2. Improving Optimizer incentive to actually visit all stations on the schedule. This is currently enforced via the 'reach final destination constraint'. To solve this, we could (hopefully xD) add a last objective line / addition that essentially checks if a train is at the final station and if not adds the delay (weighted by passengers), essentially replacing the 'reach final destination' constraint softly. Additionally, this could (and maybe should) be extended to all intermediate stations on that same route (all stations in the same direction on the last journey). To ensure that this doesn't introduce issues, schedules should be 'doable' as this then would potentially penalize too much... This has to be double-checked logically, but might be a promising approach.

In [251]:
import os
# Replace this with the actual absolute path to your .lic file in WSL
# e.g., "/home/yourusername/my_project/gurobi.lic"
os.environ["GRB_LICENSE_FILE"] = "/Users/lucas1/DataSpellProjects/TrainTransportationNetworkOptimization/gurobi.lic"

# Import Gurobi
import gurobipy as gp
from gurobipy import GRB

import numpy as np

In [252]:
# Initialize the Model
model = gp.Model("Basic_MILP")

### Adjusting the tack blueprint information

Here we also might make some modifications (like maximum allowed speed) but probably also not in this prototype...

In [253]:
# Define the track system using dictionaries to have homogeneous lists and allow for precise track building

# Copy preset for faster building
# {"type": "station", "capacity": 0}
# {"type": "segment", "length": 0, "capacity": 0}

track_blueprint = [
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 5, "capacity": 1},
    {"type": "station", "capacity": 2},
    {"type": "segment", "length": 2, "capacity": 2},
    {"type": "station", "capacity": 3}
]

In [254]:
# Parsing the track blueprint and generating necessary data (1D lists for Gurobi)
# These store the allowed number of trains on a given block and which blocks are train stations

block_capacities = []
is_station = []

for item in track_blueprint:
    if item["type"] == "station":
        block_capacities.append(item["capacity"])
        is_station.append(True)
    elif item["type"] == "segment":
        # Int conversion not necessary, but gets rid of IDE highlighting
        for _ in range(int(item["length"])):
            block_capacities.append(item["capacity"])
            is_station.append(False)
    else:
        print("typo? No other types yet...")

In [255]:
np.random.randint(0, 3)

0

### Adjusting the train schedules

Here we want to add additional train information (weights related to the number of passengers). Also, but probably not in this prototype, we want to add additional specifications like maximum speed.

**Another idea (possibly for later):** We can add random noise to the stations like

`{"station": 0, "departure": 3, "random_delay": np.random.rand(np.random.randint(0, 3))}`

to our stations and have additional constraints enforcing those to simulate a more dynamic and realistic network. The same could be done for tracks (switches). Generally, in a similar fashion (but maybe without randomness) we could introduce delays or issues into the system and see how the solver adapts the optimal solution then. 

In [256]:
# Similar to our track_blueprint, we use dictionaries for the schedules for a given train
# Especially here, this allows for basically arbitrary additional data that can be used for
# different optimizing goals and constraints. This also nicely handles, that a train doesn't
# have to stop at every station.

# Note: Due to the implementation ('looking-back') departure_times_i has to be >= 1
# Thought: Should we have an (optional) boolean 'is_destination' in (some) station
# descriptions or will this always be clear? Maybe then we could also extract this information
# using a parsing function to create lists / a matrix is_destination...
train_information = [
    # Train 0 (RE-like)
    {
        # Info can be essentially arbitrary and will likely just be used for better optimization goals
        "info": {"passenger_count": 100},
        "schedule": [
            # First journey
            {"station": 0, "departure": 3},
            {"station": 1, "arrival": 10, "departure": 13},
            # (Partially xD) Second journey
            {"station": 2, "arrival": 16, "departure": 20}, # Merged arrival / departure for the turnaround
            {"station": 1, "arrival": 27, "departure": 29},
            {"station": 0, "arrival": 34}
        ]
    },
    # Train 1 (ICE-like)
    {
        "info": {"passenger_count": 250},
        "schedule": [
            # First journey
            {"station": 0, "departure": 5},
            # (Partially xD) Second journey
            {"station": 2, "arrival": 13, "departure": 18, "dwell": 5}, # Merged arrival / departure for the turnaround
            {"station": 0, "arrival": 29}
        ]
    },
    # More trains ...
]

#### New Dataclass structure to streamline 'parameter' access later on

In [257]:
from dataclasses import dataclass
from enum import Enum

class Direction(Enum):
    fwd = "forward"
    bwd = "backward"

def tensor_for(direction: Direction):
    return x_fwd if direction is Direction.fwd else x_bwd

@dataclass(frozen=True)
class Stop:
    station_id: int
    block: int


In [258]:
# Number of time steps over which we optimize
time_horizon = 40

# Deriving some additional variables for bounding variables
num_trains = len(train_information)
num_blocks = len(block_capacities)
num_stations = sum(is_station)

# List of all block indices that are stations
station_block_indices = [block for block, station_idx in enumerate(is_station) if station_idx]

# Creating is_near_station List

In [259]:
# Use is_station list to generate is_near_station

is_near_station = [False] * num_blocks

for j in range(num_blocks):
    
    # Always mark
    if is_station[j]:
        is_near_station[j] = True

        # Check before marking to avoid out of bounds errors
        if j > 0:
            is_near_station[j - 1] = True
        if j < num_blocks - 1:
            is_near_station[j + 1] = True

### Adjusting the decision variable(s)

Instead of only having one 3D-Tensor, we will now have two (one for the forward and one for the backward direction). Using this formulation we will have to adjust most of our constraints. However, this should rather just be considering both tensors instead of actual logical changes.

In [260]:
# Define Decision Variables

# To describe our train movement we need a way to know which train is where at what
# time (-> 3D time-position grid of boolean values)

x_fwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_fwd")
x_bwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_bwd")

### Constraints (Update this)

#### General Constraints

As general constraints, we have the
- uniqueness constraint (no train can be on multiple tracks at once)
- spawning constraint (a train does not 'exist' until it leaves from the first station in its schedule). For future iterations we should also think about if we want to keep the spawning behavior or replace (or even remove?) it...
- capacity constraints (there can't exist more trains on a block (or station) than its capacity)

#### Movement Constraints

For now, we have the
- 'at most one block per timestep' constraint (will likely be changed in the next phase (step 2) to allow for different train speeds)
- 'departure-time constraint' (trains don't have to stop at stations if they are not on their schedule and also, they can't leave a station until their departure time)

In [261]:
# Testing cell for dictionary access again
i = 0
train_information[i]["schedule"][0]["departure"]

3

In [262]:
# Uniqueness constraint for trains to only exist once on a block (on only a single track at a given time)
# Additionally, we tackle the spawning (departure times), as before a train does not exist on any block
for i in range(num_trains):

    # Extract the spawn time (first departure time)
    spawn_time = train_information[i]["schedule"][0]["departure"]

    for k in range(time_horizon):

        # Sum over all blocks j to ensure that a train only exists once at a given time k
        active_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for j in range(num_blocks))

        # If the train hasn't spawned (or left) yet, it can't exist on the track and we enforce this
        if k < spawn_time:
            model.addConstr(active_tracks == 0, name=f"NotSpawned_Train{i}_Time{k}")
        else:
            # If it already exists, we enforce that the train only exists on one track at any time k
            model.addConstr(active_tracks == 1, name=f"UniquePosition_Train{i}_Time{k}")

In [263]:
# Capacity constraints (the sum over all tracks j and all trains i <= capacities_i)
for k in range(time_horizon):

    # Here we want to sum over all i AND j and add the constraint that this should be smaller than
    # Is this formulation correct and can this be made more compact (yes: x.sum('*', j, k))?
    for j in range(num_blocks):
        occupied_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for i in range(num_trains))

        # Add constraint to enforce capacity maximum across block j
        model.addConstr(occupied_tracks <= block_capacities[j], name=f"Capacity_Block{j}_Time{k}")

### Thoughts on the (current) Movement Constraints

Sadly, it seems like we're in a bit of a pickle. We wanted to update our code to allow for trains with different speeds. To do this, we first wanted to extend the back-looking approach to a larger window, e.g., for a speed of 2:

$$x_{i, j, k} + x_{i, j+1, k} \le x_{i, j, k-1} + x_{i, j-1, k-1} + x_{i, j, k-2} + x_{i, j-1, k-2}$$

This should mathematically prohibit trains from teleporting, i.e., skipping blocks if the speed is greater one. However, due to our uniqueness constraint, it HAS to teleport since otherwise it would exist on two (or generally more than one) blocks in one timestep — which is illegal. This suggests, that with our current formulation, implementing speeds greater one is impossible.

#### Alternative Formulation

A slightly less straight forward, but ultimately identical description should be to increase time resolution and impose 'block dwell times' on slower trains:

Suppose `train_0` travels with a speed of three while `train_1` only has a speed of one. Then `train_0` would be allowed in every time step, while `train_1` could only travel every third because after traveling it would have to enforce a dwell time of 2. To implement this, we could likely use a detection similar to the arrival detection via transitions in the model objective.

#### Prohibiting trains from turning around at any station

We do some fairly extensive (or verbose, at least for now) processing here to deduct the traveling direction of a train at different stations, as well as the stations where it should turn. This will also be relevant again when formulating our weighted constraints.

In [264]:
# Pretty lengthy loop to get forward and backward directions at stations to then compute number of
# stops to weigh stations with passengers (fractions of capacities).
# Might be reworked, if we see that we actually only need 'turn_indices' and not the actual directions...

# List to store direction data for trains
train_directions = []

# For all trains
for i in range(num_trains):

    # Get the schedule and maximum passenger count
    schedule = train_information[i]["schedule"]
    max_passenger_count = train_information[i]["info"]["passenger_count"]

    # List to store current direction (a (hopefully later on redundant) helper)
    current_directions = []

    # Handle spawn station direction (which actually might not even be needed)
    if schedule[0]["station"] < schedule[1]["station"]:
        current_directions.append("forward")
    elif schedule[0]["station"] > schedule[1]["station"]:
        current_directions.append("backward")
    else:
        print("First and second station in schedule seem to be identical...")

    # Loop over the schedule to compute the weights (number of passengers getting off) at each station
    # Might only explicitly give information for 'intermediate' (not spawn or final destination) stations...
    for stop_idx in range(1, len(schedule) - 1):

        # Helper variables ot determine a min / max
        previous_station_id = schedule[stop_idx - 1]["station"]
        current_station_id = schedule[stop_idx]["station"]
        next_station_id = schedule[stop_idx + 1]["station"]

        # Determine if current station is a min or max of these three values. This should give
        # is an is_turnaround indicator (or not if there are no intermediate stations?)
        if (previous_station_id < current_station_id) and (current_station_id > next_station_id):
            current_directions.append("turn")
        elif previous_station_id < current_station_id < next_station_id:
            current_directions.append("forward")
        elif previous_station_id > current_station_id > next_station_id:
            current_directions.append("backward")
        else:
            print("Unexpected case. Double check logic...")

    # Handle destination station direction
    if schedule[-2]["station"] < schedule[-1]["station"]:
        current_directions.append("forward")
    elif schedule[-2]["station"] > schedule[-1]["station"]:
        current_directions.append("backward")
    else:
        print("Last and second to last station in schedule seem to be identical...")

    # Append current_directions to train_directions
    train_directions.append(current_directions)

    print(f"Train {i}:", current_directions)

Train 0: ['forward', 'forward', 'turn', 'backward', 'backward']
Train 1: ['forward', 'turn', 'backward']


In [265]:
# Compute the block indices where a train should / is allowed to turn (given its schedule)

# Check if set comprehension or list comprehension is better here (and what each one might
# entail / restrict)

turn_blocks = []
for i in range(num_trains):
    blocks = {station_block_indices[train_information[i]["schedule"][s]["station"]]
              for s, d in enumerate(train_directions[i]) if d == "turn"}
    turn_blocks.append(blocks)

print(turn_blocks)

[{9}, {9}]


In [266]:
# Movement Constraints

# A train can (after spawning) either wait or move
for i in range(num_trains):

    # Extract the spawn station id and spawn time (first departure time)
    spawn_station_id = train_information[i]["schedule"][0]["station"]
    spawn_time = train_information[i]["schedule"][0]["departure"]

    # At every time point, we need to differentiate between different scenarios
    for k in range(time_horizon):

        # If the train hasn't left yet, it's not yet on the grid and the spawn constraint takes care of this
        if k < spawn_time:
            pass
        # Here the train spawns, and we fix a position (the block that corresponds to the first station on
        # the trains schedule)
        elif k == spawn_time:

            # Here we can again use the spawn_station_id and get the one for the next station
            block_index = station_block_indices[spawn_station_id]
            next_station_id = train_information[i]["schedule"][1]["station"]

            # Determine the initial direction of the train
            if spawn_station_id < next_station_id:
                # Moving right -> Spawn in Forward tensor
                model.addConstr(x_fwd[i, block_index, k] == 1, name=f"SpawnFwd_Train{i}_Time{k}_Block{block_index}")
            else:
                # Moving left -> Spawn in Backward tensor
                model.addConstr(x_bwd[i, block_index, k] == 1, name=f"SpawnBwd_Train{i}_Time{k}_Block{block_index}")

        else:

            # Handle out back-looking approach for both channels (tensors)
            for j in range(num_blocks):

                # Forward channel (move 'right')

                # Calculate if this block is a station where turning around is allowed (switching from Bwd to Fwd)
                turnaround_fwd = x_bwd[i, j, k - 1] if j in turn_blocks[i] else 0

                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == 0:
                    model.addConstr(x_fwd[i, 0, k] <= x_fwd[i, 0, k - 1] + turnaround_fwd, name=f"MoveFwd_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x_fwd[i, j, k] <= x_fwd[i, j, k - 1] + x_fwd[i, j - 1, k - 1] + turnaround_fwd, name=f"MoveFwd_Train{i}_Time{k}_Block{j}")

                # Backward channel (move 'left')

                # Allow switching from Fwd to Bwd (if at turn station)
                turnaround_bwd = x_fwd[i, j, k - 1] if j in turn_blocks[i] else 0

                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == num_blocks - 1:
                    model.addConstr(x_bwd[i, j, k] <= x_bwd[i, j, k - 1] + turnaround_bwd, name=f"MoveBwd_Train{i}_Time{k}_Block{num_blocks - 1}")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x_bwd[i, j, k] <= x_bwd[i, j, k - 1] + x_bwd[i, j + 1, k - 1] + turnaround_bwd, name=f"MoveBwd_Train{i}_Time{k}_Block{j}")

### Adjusting intermediate (and destination) station constraints and handling

Here we might need / want the information available if a station is a destination station. Also, here we probably need to still do some more modifications to the function and the logic...

In [267]:
# Here we add the constraints for intermediate stations (arrival and departure times and
# no spawning / final destination)

# In this version we only tackle departure times and not dwell times yet (change this?)

# For every train
for i in range(num_trains):

    # Get the trains schedule for easier access
    schedule = train_information[i]["schedule"]

    # Loop through all stops except the spawn station
    for stop_idx in range(1, len(schedule)):
        current_stop = schedule[stop_idx]

        # If there is no departure time, it's the final destination
        if "departure" not in current_stop:
            continue

        # Here we could add another check against malformed schedules (Is this good or
        # might rather cause misunderstanding?)
        if stop_idx + 1 >= len(schedule):
            print(f"Warning: Train {i} has a departure time at its final stop. Double check schedules...")
            continue

        # Extract variables
        departure_time = current_stop["departure"]
        station_id = current_stop["station"]
        station_block = station_block_indices[station_id]

        # Get the next station id to determine the travel direction
        # Here we might encounter issues with malformed schedules
        next_station_id = schedule[stop_idx + 1]["station"]

        # Handle forward direction
        if next_station_id > station_id:
            next_block = station_block + 1

            # Prohibit departure but inhibiting existence on later blocks before t_departure
            if next_block < num_blocks:
                for k in range(departure_time):
                    model.addConstr(x_fwd[i, next_block, k] == 0, name=f"DepWallFwd_Train{i}_Block{next_block}_Time{k}")

        # Handle backward direction
        if next_station_id < station_id:
            next_block = station_block - 1

            # Prohibit departure but inhibiting existence on earlier blocks before t_departure
            if next_block >= 0:
                for k in range(departure_time):
                    model.addConstr(x_bwd[i, next_block, k] == 0, name=f"DepWallBwd_Train{i}_Block{next_block}_Time{k}")

### Adjusting the Objective

Here we have to generally implement changes to use the updated data structure and also use the new information (`"info"`).

In the objective we would like to include the weights and with that, also delays at intermedite stations (also weighted). For this, I would have just for now said:

We assume that the train 'fills' its capacity at the initial station and turnaround stations and then at every station, an equal amount of people get off. This means, that if a train has 100 capacity and goes from 0 -> 1 -> 2 (here turns around) -> 0, the delay factors at the different stations are 0 (we just fill up and depart) -> 1 (50% = 50 get off since we have 2 stations) -> 2 (the other 50% = 50 get off + we 'fill up') -> 0 (all 100% = 100 get off).

We'd likely need some helper variables / lists for this, but they should be easily generatable from the train_information...

**Another though:** We could (hopefully xD) add a last objective line / addition that essentially checks if a train is at the final station and if not adds the delay (weighted by passengers), essentially replacing the 'reach final destination' constraint softly. Additionally, this could (and maybe should) be extended to all intermediate stations on that same route (all stations in the same direction on the last journey). To ensure that this doesn't introduce issues, schedules should be 'doable' as this then would protentially penalize too much... This has to be double-checked logically, but might be a promising approach.

In [268]:
# New Objective (now considering lateness at intermediate stations as well, weighted via passengers getting off)

# Old code moved to before movement constraint...

In [269]:
# Compute station weights at every station given max_passengers and train_directions

for i in range(num_trains):
    directions = train_directions[i]

    stations_till_turn = []
    turn_counter = 0

    # Compute stations until 'turn' and append count to stations_till_turn when found
    for direction in directions:
        if direction != "turn":
            turn_counter += 1
        else:
            stations_till_turn.append(turn_counter)

        # Reset turn counter (is 0 correct or should this maybe be 1?)
        turn_counter = 0

#### Update Model Objective

Here we just want to port over the old model objective to our new code base using bidirectional traveling. When this works, we can fix the other issues and extend our code and model objective. This serves as a baseline for testing.

**Regarding the 'must finish' constraint:** Currently, our schedules only have one return station (just one back and forth travel). With this, we should be able to enforce the 'must finish' constraint on `x_bwd` and be able to run the code. Since this however will not be the default, we need to update this constraint accordingly.

**Great Insight:** To mitigate the unwanted 'wait on blocks if arrival time permits' behavior, we don't have to track stations transitions to obtain arrival times. We can simply penalize BEING on tracks. Then waiting will automatically happen at stations. The only thing that still might need a solution would be late departure if the schedule (i.e. a 'relaxed' one) permits (and for 'intermediate but not visited' stations)... If this really is an issue and needs fixing, we will see later.

In [270]:
# New Objective

# Map the final station ID to the physical block index
destination_stations = [train_information[i]["schedule"][-1]["station"] for i in range(num_trains)]
destination_blocks = [station_block_indices[destination_station] for destination_station in destination_stations]

# Adding an additional constraint to ensure that a train drives to its destination station
# and not allowing the optimizer to cheese its way out of a lateness penalty (for now just
# taking care of one-time return journies — see elaboration above):

# Force every train to reach its destination by the end of the horizon
for i in range(num_trains):
    dest_block = destination_blocks[i]
    last_t = time_horizon - 1

    if train_directions[i][-1] == "forward":
        model.addConstr(x_fwd[i, dest_block, last_t] == 1, name=f"MustFinish_Train{i}_Forward")
    else:
        model.addConstr(x_bwd[i, dest_block, last_t] == 1, name=f"MustFinish_Train{i}_Backward")

# Get the target arrival times
target_arrivals = [train_information[i]["schedule"][-1]["arrival"] for i in range(num_trains)]

# List to store our terms contributing to the lateness penalty
objective_terms = []

# Loop over all trains
for i in range(num_trains):
    destination_block = destination_blocks[i]
    target_time = target_arrivals[i]

    # We only loop over k starting from the minute after the train is supposed to arrive
    # If they arrive at or before target_time, it is not penalized (adds 0)
    for k in range(target_time + 1, time_horizon):
        # Calculate the positive delay multiplier
        delay_penalty = k - target_time

        # Add a (basically negligible) 'being early bonus' to discourage the solver from
        # letting trains wait on the tracks for no reason. This should not affect the delay
        # optimization though.
        being_early_bonus = ...

        # Compute the transition value to check for arrival
        direction_tensor = x_fwd if train_directions[i][-1] == "forward" else x_bwd
        transition = direction_tensor[i, destination_block, k] - direction_tensor[i, destination_block, k - 1]

        # Multiply by the arrival transition at the destination block and add to our objective list
        objective_terms.append(delay_penalty * transition)


# Tie-Breaker wait on non-station blocks penalty
# Epsilon should be smaller num_trains * time_horizon to never account for a full 'delay unit'
epsilon = 0.5 * num_trains * time_horizon
for i in range(num_trains):
    for k in range(time_horizon):
        for j in range(num_blocks):
            # Only add if it's a non-station block
            if not is_station[j]:
                objective_terms.append(epsilon * (x_fwd[i, j, k] + x_bwd[i, j, k]))

model.setObjective(gp.quicksum(objective_terms), GRB.MINIMIZE)

In [271]:
# Run the optimization
model.optimize()

# Code by Gemini

# Print the results if an optimal solution is found
if model.Status == GRB.OPTIMAL:
    print("\n" + "="*30)
    print("🚂 OPTIMAL SCHEDULE FOUND 🚂")
    print("="*30)

    for i in range(num_trains):
        print(f"\n--- Train {i} ---")
        for k in range(time_horizon):
            for j in range(num_blocks):
                # .X extracts the solved value from the Gurobi variable.
                # We check > 0.5 to account for minor floating-point tolerances in the solver.
                if (x_fwd[i, j, k].X > 0.5) or (x_bwd[i, j, k].X > 0.5):
                    if is_station[j]:
                        print(f"Time {k:2d} -> Block {j} (Station {sum(is_station[:j])})")
                    else:
                        print(f"Time {k:2d} -> Block {j}")
else:
    print("\n❌ No optimal solution found. The model might be Infeasible.")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M3 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Academic license 2839949 - for non-commercial use only - registered to lu___@email.uni-freiburg.de
Optimize a model with 1964 rows, 1600 columns and 7484 nonzeros
Model fingerprint: 0xcf389b53
Variable types: 0 continuous, 1600 integer (1600 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]
Presolve removed 1255 rows and 1058 columns
Presolve time: 0.02s
Presolved: 709 rows, 542 columns, 2353 nonzeros
Variable types: 0 continuous, 542 integer (542 binary)
Found heuristic solution: objective 1653.0000000
Found heuristic solution: objective 1214.0000000

Root relaxation: objective 1.889678e+02, 252 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objec